In [ ]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

print("GPU доступен" if tf.config.list_physical_devices('GPU') else "Работа на CPU")

GPU доступен:


In [ ]:
# Загрузка данных
data_dir = tf.keras.utils.get_file(
    'mini_speech_commands.zip',
    origin="https://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip",
    extract=True,
    cache_dir='.', cache_subdir='data')

DATASET_PATH = os.path.join(os.path.dirname(data_dir), 'mini_speech_commands_extracted', 'mini_speech_commands')

commands = np.array(tf.io.gfile.listdir(DATASET_PATH))
commands = commands[commands != 'README.md']
print('Команды:', commands)

182082353/182082353 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Команды: ['right' 'go' 'stop' 'up' 'no' 'yes' 'left' 'down']


In [ ]:
def get_spectrogram(waveform):
    input_len = 16000
    waveform = waveform[:input_len]
    zero_padding = tf.zeros([16000] - tf.shape(waveform), dtype=tf.float32)
    waveform = tf.cast(waveform, dtype=tf.float32)
    equal_length = tf.concat([waveform, zero_padding], 0)
    spectrogram = tf.signal.stft(equal_length, frame_length=255, frame_step=128)
    spectrogram = tf.abs(spectrogram)
    spectrogram = spectrogram[..., tf.newaxis]
    return spectrogram

def decode_audio(audio_binary):
    audio, _ = tf.audio.decode_wav(contents=audio_binary)
    return tf.squeeze(audio, axis=-1)

def get_spectrogram_and_label_id(audio_file, label_id):
    audio_binary = tf.io.read_file(audio_file)
    waveform = decode_audio(audio_binary)
    spectrogram = get_spectrogram(waveform)
    spectrogram.set_shape([124, 129, 1])
    label_id.set_shape([])
    return spectrogram, label_id

def make_dataset(files, labels):
    labels = np.array(labels).astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(map_func=get_spectrogram_and_label_id, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

filenames = tf.io.gfile.glob(str(DATASET_PATH) + '/*/*')
filenames = tf.random.shuffle(filenames)

label_names = commands
label_to_id = {name: i for i, name in enumerate(label_names)}
labels = [label_to_id[os.path.basename(os.path.dirname(f.decode('utf-8')))] for f in filenames.numpy()]

val_size = int(len(filenames) * 0.2)
train_files, val_files = filenames[val_size:], filenames[:val_size]
train_labels, val_labels = labels[val_size:], labels[:val_size]

train_ds = make_dataset(train_files, train_labels)
val_ds = make_dataset(val_files, val_labels)

Датасеты пересозданы с явным указанием размерностей


In [ ]:
def residual_block(x, filters, kernel_size=3):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

input_shape = (124, 129, 1)
inputs = layers.Input(shape=input_shape)
x = layers.Rescaling(1./255)(inputs)
x = layers.Conv2D(32, 3, padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

x = residual_block(x, 32)
x = layers.MaxPooling2D()(x)
x = residual_block(x, 32)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(len(label_names), activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 124, 129,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 124, 129,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 124, 129,  │        320 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 124, 129,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 124, 129,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 124, 129,  │      9,248 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 129,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 124, 129,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 124, 129,  │      9,248 │ re_lu_1[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 124, 129,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 124, 129,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 124, 129,  │          0 │ add[0][0]         │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 62, 64,    │          0 │ re_lu_2[0][0]     │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 62, 64,    │      9,248 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 62, 64,    │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 62, 64,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 62, 64,    │      9,248 │ re_lu_3[0][0]   

 Total params: 43,208 (168.78 KB)

 Trainable params: 42,888 (167.53 KB)

 Non-trainable params: 320 (1.25 KB)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 24s 61ms/step - accuracy: 0.2548 - loss: 1.9197 - val_accuracy: 0.1225 - val_loss: 3.0652
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.3519 - loss: 1.6646 - val_accuracy: 0.1306 - val_loss: 3.9145
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 54ms/step - accuracy: 0.4155 - loss: 1.5120 - val_accuracy: 0.1456 - val_loss: 10.5613
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 22s 61ms/step - accuracy: 0.5003 - loss: 1.3492 - val_accuracy: 0.1306 - val_loss: 140.2540
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - accuracy: 0.5780 - loss: 1.1832 - val_accuracy: 0.1500 - val_loss: 8.0458
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 60ms/step - accuracy: 0.6345 - loss: 1.0512 - val_accuracy: 0.1306 - val_loss: 529.8986
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - accuracy: 0.6784 - loss: 0.9304 - val_accuracy: 0.1456 - val_loss: 32.0787
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 21s 59ms/step - accuracy: 0.7084 - loss: 0.8

In [ ]:
def make_student_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)

    # Smaller convolutional layers for ESP32 constraints
    x = layers.Conv2D(16, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.DepthwiseConv2D(3, padding='same', activation='relu')(x)
    x = layers.Conv2D(32, 1, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes)(x) # No activation here for distillation loss

    return models.Model(inputs, outputs, name='esp32_student')

student_model = make_student_model(input_shape, len(label_names))
student_model.summary()

Model: "esp32_student"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 124, 129, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 124, 129, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 62, 65, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 62, 65, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 62, 65, 16)     │           160 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 62, 65, 32)     │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 62, 65, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 31, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,320 (5.16 KB)

 Trainable params: 1,224 (4.78 KB)

 Non-trainable params: 96 (384.00 B)

In [ ]:
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher):
        super(Distiller, self).__init__()
        self.teacher = teacher
        self.student = student

    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super(Distiller, self).compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def call(self, x):
        return self.student(x)

    def train_step(self, data):
        x, y = data
        teacher_predictions = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            student_predictions = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_predictions)
            distillation_loss = self.distillation_loss_fn(
                tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
                tf.nn.softmax(student_predictions / self.temperature, axis=1)
            )
            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        y_true = tf.reshape(y, [-1])
        for metric in self.metrics:
            if metric.name != "loss" and "distillation" not in metric.name and "student" not in metric.name:
                metric.update_state(y_true, student_predictions)

        results = {m.name: m.result() for m in self.metrics}
        results.update({"loss": loss, "student_loss": student_loss, "distillation_loss": distillation_loss})
        return results

    def test_step(self, data):
        x, y = data
        y_prediction = self.student(x, training=False)
        student_loss = self.student_loss_fn(y, y_prediction)

        y_true = tf.reshape(y, [-1])
        for metric in self.metrics:
            if metric.name != "loss" and "distillation" not in metric.name and "student" not in metric.name:
                metric.update_state(y_true, y_prediction)

        results = {m.name: m.result() for m in self.metrics}
        results.update({"student_loss": student_loss})
        return results

distiller = Distiller(student=student_model, teacher=model)
distiller.compile(
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy'],
    student_loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    distillation_loss_fn=tf.keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=5
)

Дистиллятор исправлен и готов к работе


In [ ]:
distiller.fit(train_ds, epochs=10, validation_data=val_ds)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.1456 - distillation_loss: 0.0018 - loss: 0.2025 - student_loss: 2.0081 - val_loss: 0.0000e+00 - val_student_loss: 2.1887
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.1462 - distillation_loss: 0.0028 - loss: 0.2009 - student_loss: 1.9835 - val_loss: 0.0000e+00 - val_student_loss: 3.6862
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.1538 - distillation_loss: 0.0034 - loss: 0.2017 - student_loss: 1.9861 - val_loss: 0.0000e+00 - val_student_loss: 2.7566
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.1150 - distillation_loss: 0.0037 - loss: 0.2016 - student_loss: 1.9829 - val_loss: 0.0000e+00 - val_student_loss: 3.9552
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.1512 - distillation_loss: 0.0040 - loss: 0.1985 - student_loss: 1.9492 - val_loss: 0.0000e+00 - val_student_loss: 3.4287
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - ac

In [ ]:
test_results = distiller.evaluate(val_ds, return_dict=True)
# В Keras 3 метрики могут быть вложены в 'compile_metrics'
acc = test_results.get('accuracy') or test_results.get('compile_metrics', {}).get('accuracy')
loss = test_results.get('student_loss')

print(f"Точность студента на валидационной выборке: {acc:.2%}")
print(f"Loss студента: {loss:.4f}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.1962 - loss: 0.0000e+00 - student_loss: 2.3696
Точность студента на валидационной выборке: 19.62%
Loss студента: 2.3696


In [ ]:
import tensorflow as tf

# 1. Функция репрезентативного датасета для квантования
def representative_data_gen():
    # Берем небольшую выборку из валидационного набора
    for input_value, _ in val_ds.take(100):
        yield [input_value]

# 2. Настройка конвертера
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Обеспечиваем полную INT8 совместимость
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 3. Конвертация
tflite_model_quant = converter.convert()

# 4. Сохранение файла
MODEL_TFLITE = 'student_model_quant.tflite'
with open(MODEL_TFLITE, 'wb') as f:
    f.write(tflite_model_quant)

import os
size_kb = os.path.getsize(MODEL_TFLITE) / 1024
print(f'Размер квантованной модели: {size_kb:.2f} KB')

Saved artifact at '/tmp/tmphh3q7su3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 124, 129, 1), dtype=tf.float32, name='keras_tensor_23')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  137075818762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075694823248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075694824016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075694820752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075694822288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075694824976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075696083280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075696085392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075696084240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137075696085008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13707569608

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Размер квантованной модели: 8.14 KB


In [ ]:
# ESP32 требует модель в виде C-массива (HEX-дамп)
!apt-get update && apt-get install xxd
!xxd -i student_model_quant.tflite > model_data.h

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,998 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,226 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRe